In [1]:

from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL_DIR = PROJECT_ROOT / "model"
RESULTS_DIR = PROJECT_ROOT / "results"

model = keras.models.load_model(
    MODEL_DIR / "har_cnn_float32.keras"
)

norm = np.load(MODEL_DIR / "normalization.npz")
axis_mean = norm["mean"]
axis_std = norm["std"]

print("TensorFlow:", tf.__version__)
print("Parameters:", model.count_params())
print("Normalization mean:", axis_mean)
print("Normalization std:", axis_std)


TensorFlow: 2.21.0
Parameters: 2702
Normalization mean: [[[0.799536   0.02738299 0.08024956]]]
Normalization std: [[[0.41929448 0.38350964 0.37542543]]]


In [2]:

float_converter = tf.lite.TFLiteConverter.from_keras_model(model)
float_converter.optimizations = []
float_tflite = float_converter.convert()

float_path = MODEL_DIR / "har_cnn_float32.tflite"
float_path.write_bytes(float_tflite)

print(f"Float32 TFLite size: {len(float_tflite):,} bytes")


INFO:tensorflow:Assets written to: C:\Users\nandi\AppData\Local\Temp\tmpuxft7z1m\assets


INFO:tensorflow:Assets written to: C:\Users\nandi\AppData\Local\Temp\tmpuxft7z1m\assets


Saved artifact at 'C:\Users\nandi\AppData\Local\Temp\tmpuxft7z1m'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 3), dtype=tf.float32, name='accelerometer_window')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2357217525840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217529488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217528144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217528912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217531408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217532176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217527952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217531024: TensorSpec(shape=(), dtype=tf.resource, name=None)
Float32 TFLite size: 16,980 bytes


In [4]:

import numpy as np
from sklearn.model_selection import GroupShuffleSplit

DATA_ROOT = PROJECT_ROOT / "data" / "UCI HAR Dataset"

def load_split(split):
    signal_dir = DATA_ROOT / split / "Inertial Signals"

    signals = [
        np.loadtxt(
            signal_dir / f"total_acc_{axis}_{split}.txt",
            dtype=np.float32,
        )
        for axis in ["x", "y", "z"]
    ]

    X = np.stack(signals, axis=-1)
    y = np.loadtxt(
        DATA_ROOT / split / f"y_{split}.txt",
        dtype=np.int64,
    ) - 1
    subjects = np.loadtxt(
        DATA_ROOT / split / f"subject_{split}.txt",
        dtype=np.int64,
    )

    return X, y, subjects


X_train, y_train, subjects_train = load_split("train")

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, val_idx = next(
    splitter.split(X_train, y_train, groups=subjects_train)
)

X_tr = X_train[train_idx]

# Fit normalization on training subjects only.
axis_mean = X_tr.mean(axis=(0, 1), keepdims=True)
axis_std = np.maximum(
    X_tr.std(axis=(0, 1), keepdims=True),
    1e-6,
)

X_tr_norm = ((X_tr - axis_mean) / axis_std).astype(np.float32)

# Verify normalization matches the saved training parameters.
saved_norm = np.load(MODEL_DIR / "normalization.npz")

assert np.allclose(axis_mean, saved_norm["mean"])
assert np.allclose(axis_std, saved_norm["std"])

print("Representative training data:", X_tr_norm.shape)
print("Normalization matches saved parameters.")


Representative training data: (5551, 128, 3)
Normalization matches saved parameters.


In [5]:

def representative_dataset():
    count = min(300, len(X_tr_norm))

    for i in range(count):
        yield [X_tr_norm[i:i + 1].astype(np.float32)]

print("Representative windows available:", len(X_tr_norm))


Representative windows available: 5551


In [6]:

int8_converter = tf.lite.TFLiteConverter.from_keras_model(model)
int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]
int8_converter.representative_dataset = representative_dataset

# Require integer operators and integer model I/O.
int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]
int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

int8_tflite = int8_converter.convert()

int8_path = MODEL_DIR / "har_cnn_int8.tflite"
int8_path.write_bytes(int8_tflite)

print(f"INT8 TFLite size: {len(int8_tflite):,} bytes")
print("Saved:", int8_path)


INFO:tensorflow:Assets written to: C:\Users\nandi\AppData\Local\Temp\tmpm__rfiiv\assets


INFO:tensorflow:Assets written to: C:\Users\nandi\AppData\Local\Temp\tmpm__rfiiv\assets


Saved artifact at 'C:\Users\nandi\AppData\Local\Temp\tmpm__rfiiv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 3), dtype=tf.float32, name='accelerometer_window')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  2357217525840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217529488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217528144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217528912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217531408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217532176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217527952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2357217531024: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\nandi\OneDrive\Desktop\tinyml-har-esp32\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8 TFLite size: 11,328 bytes
Saved: c:\Users\nandi\OneDrive\Desktop\tinyml-har-esp32\model\har_cnn_int8.tflite


In [7]:

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score
from pathlib import Path

MODEL_DIR = PROJECT_ROOT / "model"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load official test data.
X_test, y_test, subjects_test = load_split("test")

norm = np.load(MODEL_DIR / "normalization.npz")
axis_mean, axis_std = norm["mean"], norm["std"]

X_test_norm = ((X_test - axis_mean) / axis_std).astype(np.float32)

def evaluate_tflite(model_path, X):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()

    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]

    predictions = []

    for i in range(len(X)):
        sample = X[i:i+1]

        if inp["dtype"] == np.int8:
            scale, zero_point = inp["quantization"]
            if scale <= 0:
                raise ValueError("Invalid INT8 input quantization scale.")

            sample = np.round(sample / scale + zero_point)
            sample = np.clip(sample, -128, 127).astype(np.int8)
        else:
            sample = sample.astype(inp["dtype"])

        interpreter.set_tensor(inp["index"], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(out["index"])

        if out["dtype"] == np.int8:
            scale, zero_point = out["quantization"]
            output = (output.astype(np.float32) - zero_point) * scale

        predictions.append(output[0])

    predictions = np.asarray(predictions)
    y_pred = np.argmax(predictions, axis=1)

    return y_pred, inp, out


float_path = MODEL_DIR / "har_cnn_float32.tflite"
int8_path = MODEL_DIR / "har_cnn_int8.tflite"

y_pred_float, float_in, float_out = evaluate_tflite(
    float_path, X_test_norm
)
y_pred_int8, int8_in, int8_out = evaluate_tflite(
    int8_path, X_test_norm
)

print("Float32 input/output:", float_in["dtype"], float_out["dtype"])
print("INT8 input/output:", int8_in["dtype"], int8_out["dtype"])
print("INT8 input quantization:", int8_in["quantization"])
print("INT8 output quantization:", int8_out["quantization"])

results = pd.DataFrame([
    {
        "model": "Float32 TFLite",
        "accuracy": accuracy_score(y_test, y_pred_float),
        "macro_f1": f1_score(y_test, y_pred_float, average="macro"),
        "model_size_bytes": float_path.stat().st_size,
    },
    {
        "model": "INT8 TFLite",
        "accuracy": accuracy_score(y_test, y_pred_int8),
        "macro_f1": f1_score(y_test, y_pred_int8, average="macro"),
        "model_size_bytes": int8_path.stat().st_size,
    },
])

results["accuracy_percent"] = results["accuracy"] * 100
results["macro_f1_percent"] = results["macro_f1"] * 100

display(results)

print(
    "Float32 vs INT8 prediction agreement:",
    f"{np.mean(y_pred_float == y_pred_int8) * 100:.2f}%"
)

results.to_csv(
    RESULTS_DIR / "tflite_quantization_comparison.csv",
    index=False,
)

print("Saved quantization comparison.")


Float32 input/output: <class 'numpy.float32'> <class 'numpy.float32'>
INT8 input/output: <class 'numpy.int8'> <class 'numpy.int8'>
INT8 input quantization: (0.023081965744495392, -3)
INT8 output quantization: (0.00390625, -128)


c:\Users\nandi\OneDrive\Desktop\tinyml-har-esp32\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


,model,accuracy,macro_f1,model_size_bytes,accuracy_percent,macro_f1_percent
0,Float32 TFLite,0.855786,0.856416,16980,85.578554,85.641607
1,INT8 TFLite,0.852732,0.853367,11328,85.273159,85.336674


Float32 vs INT8 prediction agreement: 96.88%
Saved quantization comparison.


In [9]:

from pathlib import Path
import numpy as np
import tensorflow as tf

firmware_main = PROJECT_ROOT / "firmware" / "esp32_har" / "main"
firmware_main.mkdir(parents=True, exist_ok=True)

# 1. Export the INT8 model as a C header.
model_bytes = (MODEL_DIR / "har_cnn_int8.tflite").read_bytes()

with open(firmware_main / "model_data.h", "w", encoding="utf-8") as f:
    f.write("#pragma once\n#include <stdint.h>\n\n")
    f.write("alignas(16) const unsigned char g_har_model[] = {\n")

    for i in range(0, len(model_bytes), 12):
        chunk = model_bytes[i:i + 12]
        f.write("    " + ", ".join(f"0x{b:02x}" for b in chunk) + ",\n")

    f.write("};\n")
    f.write(f"const unsigned int g_har_model_len = {len(model_bytes)};\n")

# 2. Load model input quantization metadata.
interpreter = tf.lite.Interpreter(
    model_path=str(MODEL_DIR / "har_cnn_int8.tflite")
)
interpreter.allocate_tensors()

input_detail = interpreter.get_input_details()[0]
assert input_detail["dtype"] == np.int8

input_scale, input_zero = input_detail["quantization"]
assert input_scale > 0
assert len(X_test_norm) == len(y_test)

# 3. Quantize all official test windows.
X_quantized = np.round(
    X_test_norm / input_scale + input_zero
)
X_quantized = np.clip(X_quantized, -128, 127).astype(np.int8)

# 4. Export binary inputs and labels.
X_path = firmware_main / "test_inputs_int8.bin"
y_path = firmware_main / "test_labels.bin"

X_quantized.tofile(X_path)
y_test.astype(np.int8).tofile(y_path)

print("Model header:", firmware_main / "model_data.h")
print("Model bytes:", len(model_bytes))
print("Test windows:", len(y_test))
print("Input shape:", X_quantized.shape)
print("Input file bytes:", X_path.stat().st_size)
print("Label file bytes:", y_path.stat().st_size)
print("Input quantization scale:", input_scale)
print("Input zero point:", input_zero)


Model header: c:\Users\nandi\OneDrive\Desktop\tinyml-har-esp32\firmware\esp32_har\main\model_data.h
Model bytes: 11328
Test windows: 2947
Input shape: (2947, 128, 3)
Input file bytes: 1131648
Label file bytes: 2947
Input quantization scale: 0.023081965744495392
Input zero point: -3
